In [1]:
import os
import pandas as pd
import numpy as np

import pickle

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, mean_squared_error

import shap


import warnings
warnings.filterwarnings('ignore')

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


# Leitura dos artefatos

In [2]:
model = pickle.load(
    open('../models/wrapped/model_pipeline_prod.pkl', 'rb')
)
model


Pipeline(steps=[('json_to_df', Json parser),
                ('seletor_1',
                 Feature Selector. Mode: train. Features: ['age', 'campaign', 'contact', 'contacts_tendency', 'default', 'education', 'job', 'marital', 'month', 'pdays', 'poutcome', 'previous', 'quarter']),
                ('cria_features', BuildFeatures()),
                ('seletor_2',
                 Feature Selector. Mode: train. Features: ['age', 'campaign', 'contact', 'contacts_te...
                               grow_policy=None, importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.03942452234009175, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=6,
                               max_leaves=None,
                               min_child_weight=45.06078916507349, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None, random_state=12, ...))])

# Classificação de promoção

## Leitura das bases (treino e teste)

In [3]:
df_treino = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train.csv'))

print(df_treino.shape)
df_treino.head()

(32940, 23)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,quarter,contacts_tendency,y
0,31,admin.,single,university.degree,no,no,no,cellular,sep,fri,...,0,nonexistent,-3.4,92.379,-29.8,0.803,5017.5,3Q,0.0,no
1,39,housemaid,married,high.school,no,yes,no,telephone,may,fri,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,2Q,0.0,no
2,34,entrepreneur,married,professional.course,no,yes,no,cellular,jul,thu,...,0,nonexistent,1.4,93.918,-42.7,4.958,5228.1,3Q,0.0,no
3,36,technician,married,basic.9y,unknown,no,no,telephone,may,tue,...,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,2Q,0.0,no
4,25,student,single,unknown,unknown,yes,no,cellular,aug,fri,...,0,nonexistent,-2.9,92.201,-31.4,0.825,5076.2,3Q,0.0,no


In [4]:
df_teste = pd.read_csv(os.path.join('..', 'data', 'train_test', 'test.csv'))

print(df_teste.shape)
df_teste.head()

(8236, 23)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,quarter,contacts_tendency,y
0,27,student,single,high.school,unknown,yes,no,telephone,may,thu,...,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0,2Q,0.0,no
1,60,admin.,divorced,professional.course,no,yes,no,cellular,sep,wed,...,1,failure,-1.1,94.199,-37.5,0.886,4963.6,3Q,1.0,yes
2,51,blue-collar,married,high.school,no,no,no,telephone,may,fri,...,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,2Q,0.0,no
3,39,blue-collar,single,unknown,unknown,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,2Q,0.0,no
4,35,blue-collar,married,basic.9y,unknown,no,no,telephone,may,fri,...,0,nonexistent,1.1,93.994,-36.4,4.864,5191.0,2Q,0.0,no


In [5]:
seletor = pickle.load(
    open(os.path.join('..', 'models', 'encoders', 'seletor_2.pkl'), 'rb')
)

## Scoring

In [6]:
df_treino['score'] = (model.predict_proba(df_treino)[:,1]*1000).astype(int)
df_teste['score'] = (model.predict_proba(df_teste)[:,1]*1000).astype(int)

## Evaluation

In [7]:
roc_auc_score(
    df_treino['y'],
    df_treino['score']/1000
)

0.7826572732300403

In [8]:
roc_auc_score(
    df_teste['y'],
    df_teste['score']/1000
)

0.7574778356152653

## Análise de resultados

Para analisar os resultados obtidos do modelo de classificação, optou-se por trabalhar com o score no intervalo inteiro de $[0, 1000]$. Posteriormente, quebrou-o em decis e, então, calculou o recall (FPR) e a precisão (ou hit rate). Note que, para o decil 8 há, aproximadamente, 70% das promoções e 40% de precisão, para uma região em que há somente 22% de produtos que não deveriam ser promoções. Assim, considerou-se que o modelo é bastante satisfatório.

Isso significa que, com o auxílio do modelo, a pessoa responsável por propor as promoções poderia concentrar sua análise em apenas 20% dos produtos, ao invés de analisar a totalidade, e ainda assim capturar 70% dos produtos promocionais nesse grupo prioritário.

In [9]:
df_teste['decil'] = pd.qcut(df_teste['score'], q=10, labels=False, duplicates='drop')
results = (df_teste.groupby('decil').y.sum()/df_teste.y.sum()).reset_index()
results['decil'] += 1

results

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
results['promo_items'] = df_teste[df_teste.y == 1].groupby('decil').size()
results['total_items'] = df_teste.groupby('decil').size().astype(int)
results['no_promo_items'] = results['total_items'] - results['promo_items']
results['score (>=)'] = round(df_teste.groupby('decil').score.min(), 3)

results.fillna(0, inplace=True)
results.sort_values(by='decil', ascending=False, inplace=True)
results = results[['decil', 'score (>=)', 'promo_items', 'no_promo_items', 'total_items', 'y']]

In [ ]:
results = results.rename(columns={'y': 'recall'})

results['recall'] = results['recall'].cumsum()
results['precision'] = results['promo_items'] / results['total_items']
results['no_promo_items'] = results['no_promo_items'].cumsum()
results['total_items'] = results['total_items'].cumsum()

results.reset_index(drop=True, inplace=True)

results


Em relação à importância das features, o preço real do produto (`price_tratado`) se destaca, seguido pelo valor mínimo da parcela do produto (`installments_price`). Por outro lado, os valores SHAP, que utilizam uma abordagem diferente para calcular a importância das features em comparação com a métrica de importância gerada pelo próprio modelo, apontam que a elegibilidade do produto (`sale_price_condition_eligible`) é a features mais relevante, enquanto a aceitação do Mercado Pago (accepts_mercadopago) ocupa a segunda posição como features de maior importância.

In [ ]:
importance = model[-1].get_booster().get_score(importance_type = 'weight')

df_imp = pd.DataFrame({
    'feature': seletor.features,
    'imp': importance.values()
}).sort_values(by='imp', ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(df_imp['feature'], df_imp['imp'], color = 'skyblue')
plt.xlabel('Importância')
plt.ylabel('Features')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.grid(axis = 'x', linestyle = '--', alpha = 0.6)

plt.tight_layout()
plt.show()

In [ ]:
df_treino_encoded = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train_encoded.csv'))
sample_train = df_treino_encoded[seletor.features].sample(frac = 0.3, random_state=98)

In [ ]:
explainer = shap.Explainer(model[-1], sample_train)
shap_values = explainer(sample_train)

plt.figure(figsize = (10, 6))
shap.summary_plot(shap_values, sample_train)